In [2]:
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [7]:
dirpath = os.getcwd()
features_path = r"C:\Users\marie\rep_codes\udder_project\udder_processing\features_dict\gmfeature_table.csv"
data_path = r"C:\Users\marie\rep_codes\udder_project\udder_analysis\long_format_df"
visit_path = r"C:\Users\marie\rep_codes\udder_project\delpro_vms\data\milk_videos_visit.csv"
plot_dir = os.path.join(os.path.normpath(dirpath + os.sep + os.pardir),r"adsa\examples")

In [20]:
df = pd.read_csv(os.path.join(data_path, "lactation_features.csv"))
vdf = pd.read_csv(visit_path)
vdf_selected = vdf[['cow', 'days_in_milk']]
df_merged = df.merge(vdf_selected, on = 'cow')

In [22]:
# add min teat length, max teat length, min eu distance, max eu distance, min gd distance, max gd distance
df_merged["min_teat"] = [np.nanmin(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["max_teat"]= [np.nanmax(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["min_eu"] = [np.nanmin(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_eu"] = [np.nanmax(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_gd"] = [np.nanmax(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]

C:\Users\marie\AppData\Local\Temp\ipykernel_18676\1974453648.py:2: RuntimeWarning: All-NaN slice encountered
  df_merged["min_teat"] = [np.nanmin(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_18676\1974453648.py:3: RuntimeWarning: All-NaN slice encountered
  df_merged["max_teat"]= [np.nanmax(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_18676\1974453648.py:6: RuntimeWarning: All-NaN slice encountered
  df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_18676\1974453648.py:6: RuntimeWarning: All-NaN slice encountered
  df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float'))

In [26]:
udder_features = ['vol_udder', 'sarea_udder', 'peri_udder', 'area_udder', 'circ_udder', 'exc_udder','min_teat', 'max_teat', 'min_eu', 'max_eu', 'min_gd', 'max_gd']
prod_vars = ['yield_visit_mean', 'interval_sec_mean', 'kickoff_any_perc', 'days_in_milk', "lactation"]

In [27]:
udder_pearson_df = pd.DataFrame(index = udder_features, columns = prod_vars)
udder_pvals_df = pd.DataFrame(index = udder_features, columns = prod_vars)

In [28]:
for u in udder_features:
    for v in prod_vars:
        selected = df_merged[[v, u]].dropna(axis=0) 
        res = stats.pearsonr(selected[u], selected[v])
        udder_pearson_df.loc[u, v] = np.round(res.statistic, 3)
        udder_pvals_df.loc[u, v] = np.round(res.pvalue, 3)

In [29]:
udder_pearson_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.45,0.114,-0.078,-0.01,0.295
sarea_udder,0.464,-0.047,-0.136,-0.056,0.384
peri_udder,0.562,-0.072,-0.229,0.017,0.724
area_udder,0.546,-0.073,-0.205,-0.019,0.7
circ_udder,-0.361,0.024,0.059,-0.078,-0.358
exc_udder,0.031,0.065,0.026,-0.192,0.079
min_teat,0.202,-0.033,-0.026,0.045,0.313
max_teat,0.133,-0.026,-0.01,-0.02,0.161
min_eu,0.139,0.098,0.15,-0.131,0.12
max_eu,0.378,-0.046,-0.169,0.052,0.48


In [30]:
udder_pvals_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.0,0.195,0.374,0.913,0.001
sarea_udder,0.0,0.589,0.118,0.518,0.0
peri_udder,0.0,0.406,0.007,0.846,0.0
area_udder,0.0,0.397,0.017,0.823,0.0
circ_udder,0.0,0.789,0.517,0.391,0.0
exc_udder,0.731,0.465,0.772,0.03,0.378
min_teat,0.017,0.695,0.757,0.592,0.0
max_teat,0.117,0.762,0.903,0.815,0.056
min_eu,0.098,0.244,0.075,0.119,0.154
max_eu,0.0,0.583,0.045,0.537,0.0


In [32]:
udder_pvals_df.to_csv(os.path.join("tables", "udder_pvals_df.csv"), index = True)
udder_pearson_df.to_csv(os.path.join("tables", "udder_pearson_df.csv"), index = True
                       )